# Laboratorio 3 - Atención en BERT

**Integrantes:** Sergio Orellana (221122) y Rodrigo Mansilla (22611)  
**Modelo:** `bert-base-multilingual-cased`  
**Corpus:** *La metamorfosis*, Franz Kafka

| Parte | Responsable / estado |
|---|---|
| A. Tokenización | Sergio - completa y ejecutada |
| B. Ejecución del modelo | Rodrigo - espacio pendiente |
| C. Análisis de atención | Sergio - completa y ejecutada |
| D. Comparación | Rodrigo - espacio pendiente |

> Nota metodológica: los pesos de atención describen distribuciones internas del modelo, pero por sí solos no demuestran importancia causal ni constituyen una explicación completa de la predicción.

## 0. Reproducibilidad y selección del corpus

Elegimos **dos páginas consecutivas al azar** entre las páginas de contenido. Usamos la semilla `2009`, que fija la selección en las páginas 20 y 21 del archivo PDF (índices humanos del documento digital). Guardamos exactamente esas dos páginas en `corpus_paginas_20_21.pdf`.

Para mantener legible nuestro análisis, tomamos una oración de cada página:

1. Una oración compleja de la página 20.
2. Una oración simple de la página 21.

Cotejamos visualmente las transcripciones contra el PDF y conservamos las tildes y los signos de puntuación.

In [1]:
from pathlib import Path
import os
import random
import re

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

import pandas as pd
import torch
import transformers
from IPython.display import display, Markdown
from transformers import AutoModel, AutoTokenizer

SEED = 2009
random.seed(SEED)
torch.manual_seed(SEED)

pagina_inicial = random.Random(SEED).randint(3, 42)
assert pagina_inicial == 20

MODELO = "bert-base-multilingual-cased"
ORACIONES = {
    "P20-compleja": (
        "Así pudo enterarse Gregorio, con gran satisfacción -el padre se extendía en sus "
        "explicaciones, pues hacía tiempo que no se había ocupado de aquellos asuntos, y además "
        "la madre tardaba en entenderlos- que, a pesar de la desgracia, les había quedado algún "
        "dinero; no mucho, desde luego, pero poco a poco había ido aumentando desde entonces, "
        "gracias a los intereses intactos."
    ),
    "P21-simple": "Poco a poco empezó a ver con menos claridad.",
}

print(f"Python/Torch/Transformers: {torch.__version__} / {transformers.__version__}")
print(f"Selección reproducible: páginas {pagina_inicial} y {pagina_inicial + 1}")
pd.DataFrame(
    [{"oración": clave, "texto": texto} for clave, texto in ORACIONES.items()]
)

Python/Torch/Transformers: 2.14.0+cpu / 4.57.6
Selección reproducible: páginas 20 y 21


,oración,texto
0,P20-compleja,"Así pudo enterarse Gregorio, con gran satisfac..."
1,P21-simple,Poco a poco empezó a ver con menos claridad.


## Parte A - Tokenización

Cargamos el tokenizador *cased* multilingüe de BERT. Para cada oración mostramos la posición, el token y su tipo. Observamos que `[CLS]` abre la secuencia y `[SEP]` la cierra. Interpretamos los tokens que comienzan con `##` como continuaciones de una palabra dividida en subpalabras.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODELO, use_fast=True)


def clasificar_token(token):
    if token in tokenizer.all_special_tokens:
        return "especial"
    if token.startswith("##"):
        return "continuación de subpalabra"
    if re.fullmatch(r"[^\w]+", token, flags=re.UNICODE):
        return "puntuación"
    return "palabra o inicio de palabra"


tablas_tokens = {}
resumen_a = []

for nombre, oracion in ORACIONES.items():
    codificacion = tokenizer(oracion, return_offsets_mapping=True)
    tokens = tokenizer.convert_ids_to_tokens(codificacion["input_ids"])
    tabla = pd.DataFrame({
        "posición": range(len(tokens)),
        "token": tokens,
        "tipo": [clasificar_token(t) for t in tokens],
        "offset": [str(tuple(x)) for x in codificacion["offset_mapping"]],
    })
    tablas_tokens[nombre] = tabla

    palabras = re.findall(r"\b\w+\b", oracion, flags=re.UNICODE)
    resumen_a.append({
        "oración": nombre,
        "palabras lingüísticas": len(palabras),
        "tokens totales (con especiales)": len(tokens),
        "continuaciones ##": sum(t.startswith("##") for t in tokens),
        "tokens especiales": ", ".join(t for t in tokens if t in tokenizer.all_special_tokens),
    })

    display(Markdown(f"### {nombre}"))
    display(tabla)

display(Markdown("### Resumen de tokenización"))
display(pd.DataFrame(resumen_a))

### P20-compleja

,posición,token,tipo,offset
0,0,[CLS],especial,"(0, 0)"
1,1,Así,palabra o inicio de palabra,"(0, 3)"
2,2,pudo,palabra o inicio de palabra,"(4, 8)"
3,3,enter,palabra o inicio de palabra,"(9, 14)"
4,4,##arse,continuación de subpalabra,"(14, 18)"
...,...,...,...,...
82,82,intereses,palabra o inicio de palabra,"(353, 362)"
83,83,intact,palabra o inicio de palabra,"(363, 369)"
84,84,##os,continuación de subpalabra,"(369, 371)"
85,85,.,puntuación,"(371, 372)"


### P21-simple

,posición,token,tipo,offset
0,0,[CLS],especial,"(0, 0)"
1,1,Poco,palabra o inicio de palabra,"(0, 4)"
2,2,a,palabra o inicio de palabra,"(5, 6)"
3,3,poco,palabra o inicio de palabra,"(7, 11)"
4,4,empezó,palabra o inicio de palabra,"(12, 18)"
5,5,a,palabra o inicio de palabra,"(19, 20)"
6,6,ver,palabra o inicio de palabra,"(21, 24)"
7,7,con,palabra o inicio de palabra,"(25, 28)"
8,8,menos,palabra o inicio de palabra,"(29, 34)"
9,9,clar,palabra o inicio de palabra,"(35, 39)"


### Resumen de tokenización

,oración,palabras lingüísticas,tokens totales (con especiales),continuaciones ##,tokens especiales
0,P20-compleja,61,87,12,"[CLS], [SEP]"
1,P21-simple,9,13,1,"[CLS], [SEP]"


### Observaciones de la parte A

- Observamos que `[CLS]` y `[SEP]` no pertenecen al texto, sino que BERT los añade como tokens especiales.
- Comprobamos que el número de tokens del modelo no coincide con el número de palabras lingüísticas. La puntuación ocupa posiciones propias y algunas palabras se dividen en una pieza inicial y continuaciones `##`.
- Cuando encontramos una palabra dividida, consideramos que cada subpalabra recibe su propio vector y su propia fila y columna en la matriz de atención. Por eso no interpretamos una sola pieza como si representara automáticamente toda la palabra. Si quisiéramos analizar palabras completas, agregaríamos sus piezas mediante una regla explícita, como una suma o un promedio.

## Preparación técnica compartida para la parte C

Para desarrollar la parte C necesitamos que el modelo produzca atenciones. En esta celda realizamos únicamente la preparación computacional común para que nuestro análisis sea reproducible. **Dejamos pendiente para Rodrigo el reporte solicitado en la parte B.**

In [3]:
modelo = AutoModel.from_pretrained(
    MODELO,
    output_attentions=True,
    attn_implementation="eager",
)
modelo.eval()

ejecuciones = {}
with torch.no_grad():
    for nombre, oracion in ORACIONES.items():
        entradas = tokenizer(oracion, return_tensors="pt", return_offsets_mapping=True)
        offsets = entradas.pop("offset_mapping")[0]
        outputs = modelo(**entradas, output_attentions=True)
        ejecuciones[nombre] = {
            "oracion": oracion,
            "entradas": entradas,
            "offsets": offsets,
            "tokens": tokenizer.convert_ids_to_tokens(entradas["input_ids"][0]),
            "attentions": tuple(a.cpu() for a in outputs.attentions),
        }

print(f"Atenciones calculadas para {len(ejecuciones)} oraciones; el modelo permanece en modo evaluación.")

Atenciones calculadas para 2 oraciones; el modelo permanece en modo evaluación.


## Parte B - Ejecución del modelo (pendiente: Rodrigo)

En esta sección completaremos, sin modificar las partes A y C, los siguientes puntos:

1. Explicaremos cómo solicitamos las atenciones (`output_attentions=True`).
2. Reportaremos cuántas capas y cabezas tiene el modelo.
3. Mostraremos y explicaremos la forma de las matrices para cada oración: `(batch_size, número_de_cabezas, número_de_tokens, número_de_tokens)`.
4. Comentaremos por qué las dos últimas dimensiones cambian cuando cambia la tokenización de la oración.

In [ ]:
# TODO (Rodrigo): completar y ejecutar la Parte B.
# Sugerencia: inspeccionar len(ejecuciones[nombre]["attentions"])
# y ejecuciones[nombre]["attentions"][0].shape para cada oración.

## Parte C - Análisis de atención

Inspeccionamos dos capas alejadas (**1 y 12**) y dos cabezas (**1 y 7**) en cada capa. Elegimos dos palabras relevantes de cada oración:

- `P20-compleja`: **Gregorio** y **dinero**.
- `P21-simple`: **empezó** y **claridad**.

Si una palabra objetivo se divide, usamos como consulta su primera subpalabra y registramos la segmentación completa. Para cada combinación mostramos los cinco tokens con mayor peso de atención, sin eliminar tokens especiales, puntuación ni autoatención.

In [5]:
OBJETIVOS = {
    "P20-compleja": ["Gregorio", "dinero"],
    "P21-simple": ["empezó", "claridad"],
}
CAPAS = [0, 11]     # índices base 0 -> capas humanas 1 y 12
CABEZAS = [0, 6]    # índices base 0 -> cabezas humanas 1 y 7


def indices_de_palabra(texto, palabra, offsets):
    inicio = texto.casefold().index(palabra.casefold())
    fin = inicio + len(palabra)
    return [
        i for i, (a, b) in enumerate(offsets.tolist())
        if b > a and a < fin and b > inicio
    ]


filas = []
for nombre, datos in ejecuciones.items():
    for palabra in OBJETIVOS[nombre]:
        indices = indices_de_palabra(datos["oracion"], palabra, datos["offsets"])
        if not indices:
            raise ValueError(f"No se localizaron subpalabras para {palabra!r}")
        indice_consulta = indices[0]
        segmentacion = " + ".join(datos["tokens"][i] for i in indices)

        for capa in CAPAS:
            for cabeza in CABEZAS:
                pesos = datos["attentions"][capa][0, cabeza, indice_consulta]
                valores, posiciones = torch.topk(pesos, k=min(5, len(pesos)))
                fila = {
                    "oración": nombre,
                    "objetivo": palabra,
                    "subpalabra consulta": datos["tokens"][indice_consulta],
                    "segmentación completa": segmentacion,
                    "capa": capa + 1,
                    "cabeza": cabeza + 1,
                }
                for rango, (posicion, valor) in enumerate(zip(posiciones.tolist(), valores.tolist()), start=1):
                    fila[f"top {rango}"] = f"{datos['tokens'][posicion]} ({valor:.4f})"
                filas.append(fila)

resultados_c = pd.DataFrame(filas)
display(resultados_c)

# Evidencia tabular reutilizable fuera del notebook.
resultados_c.to_csv("resultados_atencion_parte_c.csv", index=False, encoding="utf-8-sig")
print("Tabla guardada en resultados_atencion_parte_c.csv")

,oración,objetivo,subpalabra consulta,segmentación completa,capa,cabeza,top 1,top 2,top 3,top 4,top 5
0,P20-compleja,Gregorio,Gregorio,Gregorio,1,1,Gregorio (0.2162),padre (0.0725),[SEP] (0.0371),Así (0.0336),madre (0.0286)
1,P20-compleja,Gregorio,Gregorio,Gregorio,1,7,", (0.2146)",[CLS] (0.1801),##is (0.0588),enter (0.0504),con (0.0453)
2,P20-compleja,Gregorio,Gregorio,Gregorio,12,1,Gregorio (0.2746),##arse (0.2309),[CLS] (0.0897),[SEP] (0.0725),", (0.0704)"
3,P20-compleja,Gregorio,Gregorio,Gregorio,12,7,Gregorio (0.2453),", (0.1556)",##arse (0.1508),[CLS] (0.1010),- (0.0684)
4,P20-compleja,dinero,dinero,dinero,1,1,dinero (0.0928),enter (0.0496),[CLS] (0.0321),[SEP] (0.0273),pues (0.0230)
5,P20-compleja,dinero,dinero,dinero,1,7,algún (0.3609),; (0.1815),dinero (0.0637),no (0.0553),mucho (0.0496)
6,P20-compleja,dinero,dinero,dinero,12,1,dinero (0.1304),intereses (0.1034),##los (0.0778),padre (0.0435),los (0.0431)
7,P20-compleja,dinero,dinero,dinero,12,7,dinero (0.3460),intereses (0.1008),la (0.0471),queda (0.0447),explica (0.0351)
8,P21-simple,empezó,empezó,empezó,1,1,[CLS] (0.4054),[SEP] (0.1571),poco (0.0923),Poco (0.0702),empezó (0.0686)
9,P21-simple,empezó,empezó,empezó,1,7,poco (0.3580),a (0.1521),[CLS] (0.0963),a (0.0685),ver (0.0679)


Tabla guardada en resultados_atencion_parte_c.csv


In [6]:
# Medimos cuánto se parecen los conjuntos top-5 al cambiar capa o cabeza.
def conjunto_top5(fila):
    return {fila[f"top {i}"].rsplit(" (", 1)[0] for i in range(1, 6)}


comparaciones = []
for (oracion, objetivo), grupo in resultados_c.groupby(["oración", "objetivo"], sort=False):
    registros = list(grupo.to_dict("records"))
    for i, a in enumerate(registros):
        for b in registros[i + 1:]:
            cambia_una_dimension = (a["capa"] == b["capa"]) ^ (a["cabeza"] == b["cabeza"])
            if cambia_una_dimension:
                ca, cb = conjunto_top5(a), conjunto_top5(b)
                comparaciones.append({
                    "oración": oracion,
                    "objetivo": objetivo,
                    "configuración A": f"C{a['capa']}-H{a['cabeza']}",
                    "configuración B": f"C{b['capa']}-H{b['cabeza']}",
                    "coincidencias top-5": len(ca & cb),
                    "Jaccard": len(ca & cb) / len(ca | cb),
                })

comparaciones_c = pd.DataFrame(comparaciones)
display(comparaciones_c.round({"Jaccard": 3}))
print(f"Jaccard medio entre configuraciones: {comparaciones_c['Jaccard'].mean():.3f}")

,oración,objetivo,configuración A,configuración B,coincidencias top-5,Jaccard
0,P20-compleja,Gregorio,C1-H1,C1-H7,0,0.000
1,P20-compleja,Gregorio,C1-H1,C12-H1,2,0.250
2,P20-compleja,Gregorio,C1-H7,C12-H7,2,0.250
3,P20-compleja,Gregorio,C12-H1,C12-H7,4,0.667
4,P20-compleja,dinero,C1-H1,C1-H7,1,0.111
5,P20-compleja,dinero,C1-H1,C12-H1,1,0.111
6,P20-compleja,dinero,C1-H7,C12-H7,1,0.111
7,P20-compleja,dinero,C12-H1,C12-H7,2,0.250
8,P21-simple,empezó,C1-H1,C1-H7,2,0.286
9,P21-simple,empezó,C1-H1,C12-H1,3,0.429


Jaccard medio entre configuraciones: 0.324


## Parte D - Comparación (pendiente: Rodrigo)

En esta sección compararemos `P20-compleja` con `P21-simple` usando `resultados_c` y `comparaciones_c`, sin modificar las partes A y C:

- Compararemos los receptores y pesos de ambas oraciones.
- Explicaremos qué cambia al variar el contexto y la longitud.
- Ampliaremos la respuesta 5 con la interpretación de Rodrigo.
- Distinguiremos las observaciones de la tabla de nuestras interpretaciones lingüísticas.
- Cerraremos con las limitaciones de usar atención como explicación.

**Respuesta de Rodrigo:**

_Aquí escribiremos el análisis adicional de Rodrigo para la parte D._

### Respuestas a las preguntas de análisis

1. **¿Qué tokens reciben mayor atención desde cada token seleccionado?** Para **Gregorio** observamos atención hacia el propio token, "padre", "madre", "enter", "##arse", signos y "[CLS]"/"[SEP]", según la configuración. Desde **dinero** destacan el propio token, "algún", "intereses", "padre", "no", "mucho", "queda" y signos. Desde **empezó** aparecen "Poco"/"poco", "a", "ver", el propio token, el punto y los especiales. Desde la primera pieza de **claridad** ("clar") aparecen "##idad", "con", "empezó", "poco", el punto y los especiales. En la tabla reportamos los cinco receptores exactos y sus pesos para cada combinación.

2. **¿Cambian los patrones entre capas?** Sí. Comprobamos que las capas 1 y 12 no conservan el mismo conjunto top-5. Por ejemplo, desde "Gregorio" en la cabeza 1, la capa 1 asigna su segundo mayor peso a "padre" (0.0725), mientras que la capa 12 lo asigna a "##arse" (0.2309). Desde "empezó", la capa 1 cabeza 1 se concentra primero en "[CLS]" (0.4054), pero la capa 12 cabeza 1 se concentra primero en el punto (0.2213) y luego en "Poco" y "ver". El Jaccard medio de 0.324 confirma que el solapamiento entre conjuntos top-5 es limitado.

3. **¿Cambian los patrones entre cabezas?** Sí. Dentro de la capa 1, la cabeza 1 de "dinero" reparte su mayor peso al propio token (0.0928), mientras que la cabeza 7 se concentra en "algún" (0.3609) y el punto y coma (0.1815). Para "clar", la cabeza 1 distribuye la atención entre "clar", "con", "empezó", "##idad" y "poco"; la cabeza 7 asigna 0.5638 a "##idad". Concluimos que las cabezas capturan relaciones diferentes, aunque no podemos asignarles una función lingüística universal a partir de estos dos ejemplos.

4. **¿Las palabras con mayor atención son lingüísticamente relevantes?** Observamos una mezcla. "Algún", "dinero", "intereses", "padre", "Poco", "ver", "con" y la relación "clar"-"##idad" permiten proponer vínculos léxicos o estructurales plausibles. Sin embargo, también reciben atención alta la puntuación, "[CLS]", "[SEP]" y la autoatención. Por ello consideramos la relevancia caso por caso y no equiparamos automáticamente un peso alto con importancia lingüística.

5. **¿Qué diferencias aparecen entre oraciones simples y complejas?** Nuestra oración compleja produjo 87 tokens y la simple 13. En la oración simple observamos distribuciones más concentradas en algunos casos, como 0.4054 hacia "[CLS]" desde "empezó" y 0.5638 desde "clar" hacia "##idad". En la compleja, el peso se reparte entre muchas más posiciones y aparecen asociaciones con elementos alejados, como "dinero" con "intereses". Interpretamos esta diferencia con cautela: la longitud y la estructura amplían los posibles receptores, pero una sola pareja de oraciones no permite generalizar a todas las oraciones simples y complejas.

6. **¿Qué ocurre cuando una palabra se divide en subpalabras?** Comprobamos que cada pieza ocupa una posición independiente y recibe su propia fila y columna de atención. BERT dividió, por ejemplo, "claridad" en "clar + ##idad"; desde "clar", la capa 1 cabeza 7 asignó 0.5638 a "##idad". También observamos divisiones como "enter + ##arse" e "intact + ##os". Para comparar palabras completas tendríamos que agregar explícitamente los pesos de sus piezas; en este laboratorio declaramos que usamos la primera pieza como token de consulta.

7. **¿Qué no podemos concluir observando únicamente los pesos de atención?** No podemos demostrar causalidad, importancia global, razonamiento, correferencia correcta ni una explicación fiel de una predicción. Tampoco podemos afirmar que una cabeza tenga siempre la misma función lingüística. Para sostener conclusiones causales necesitaríamos intervenciones, ablaciones u otros métodos complementarios.